# Notebook 01: Data Collection

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li  

## Overview

This notebook handles data acquisition. The primary dataset is the Boston Building and Property Violations dataset, which is publicly available via Boston's Open Data portal. We also document the neighborhood population data sourced from the U.S. Census Bureau, which will be used in Notebook 04 for normalization.

**Data sources:**
- Boston Building & Property Violations: https://data.boston.gov/dataset/building-and-property-violations1
- Boston Neighborhoods shapefile: https://data.boston.gov/dataset/boston-neighborhoods
- Neighborhood population estimates: U.S. Census Bureau ACS 2020 5-Year Estimates

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path

# Project root relative to this notebook
PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'

for d in [RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directory structure verified.')
print(f'Raw data dir: {RAW_DIR}')

## 1. Load the Violations Dataset

The raw CSV was downloaded from the Boston Open Data portal and placed in `data/raw/violations.csv`. We load and inspect it here.

In [ ]:
violations_path = RAW_DIR / 'violations.csv'
df = pd.read_csv(violations_path, low_memory=False)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2. Initial Data Inspection

We check column data types, missing value rates, and the date range covered by the dataset.

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print()
print('--- Missing Values (%) ---')
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct[missing_pct > 0])

In [ ]:
# Parse the status date column
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
valid_dates = df['status_dttm'].dropna()
print(f'Date range: {valid_dates.min()} → {valid_dates.max()}')
print(f'Records with valid date: {valid_dates.shape[0]:,} of {len(df):,}')

## 3. Neighborhood Population Data (External)

We embed approximate 2020 Census population estimates per Boston neighborhood for normalization in Notebook 04. These figures are sourced from the Boston Planning & Development Agency (BPDA) neighborhood profiles and the ACS 2020 5-Year Estimates.

In [ ]:
# Approximate 2020 ACS population by Boston neighborhood
neighborhood_population = {
    'Allston':           28806,
    'Back Bay':          22004,
    'Bay Village':        1521,
    'Beacon Hill':        9957,
    'Brighton':          46665,
    'Charlestown':       18102,
    'Chinatown':          9325,
    'Dorchester':       121779,
    'Downtown':          10291,
    'East Boston':       45782,
    'Fenway':            39167,
    'Hyde Park':         34009,
    'Jamaica Plain':     40618,
    'Mattapan':          36797,
    'Mission Hill':      15978,
    'North End':         10695,
    'Roslindale':        32961,
    'Roxbury':           59882,
    'South Boston':      36154,
    'South End':         35831,
    'West End':           4926,
    'West Roxbury':      32101,
    'Leather District':   1302,
    'Longwood':           2534,
    'Harbor Islands':      158,
}

pop_df = pd.DataFrame(list(neighborhood_population.items()),
                      columns=['neighborhood', 'population_2020'])
pop_df.to_csv(EXTERNAL_DIR / 'neighborhood_population.csv', index=False)
print('Saved neighborhood population data.')
pop_df.head(10)

## 4. Unique Values in Key Columns

We examine the range of violation codes, statuses, and cities to understand what the dataset covers.

In [ ]:
print('Unique statuses:', df['status'].unique())
print(f'Unique violation codes: {df["code"].nunique()}')
print(f'Unique violation descriptions: {df["description"].nunique()}')
print('Top 10 cities in data:')
print(df['violation_city'].value_counts().head(10))

## 5. Summary

Key findings from data collection:
- The dataset contains **~17,000 violation records** spanning multiple years.
- Key fields include violation code, description, city/neighborhood, status, and geo-coordinates.
- A meaningful fraction of records lack a `status_dttm` (date), which we address in Notebook 02.
- External population data has been saved to `data/external/neighborhood_population.csv` for later normalization.

Proceed to **Notebook 02: Data Cleaning**.

In [ ]:
# Save a summary metadata file
summary = {
    'total_records': int(len(df)),
    'columns': df.columns.tolist(),
    'date_min': str(valid_dates.min()),
    'date_max': str(valid_dates.max()),
    'unique_codes': int(df['code'].nunique()),
}
with open(PROCESSED_DIR / 'dataset_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved to data/processed/dataset_summary.json')